In [1]:
print("Hello, testing output")

Hello, testing output


# Generalizability Evaluation for Belief Tracking Repository

## Overview
This notebook evaluates the generalizability of findings in the belief-tracking research repository.

**Repository path**: `/net/scratch2/smallyan/belief_tracking_eval`

## Research Summary:
The original work investigates how language models (Llama-3-70B-Instruct, Llama-3.1-405B-Instruct) track character beliefs using a "lookback mechanism". Key findings:
- Answer payload localizes at layers 56+ at final token (80 layers total)
- Answer pointer at layers 34-52 at final token
- Binding address/payload at layers 33-38 at state token
- Binding source reference at layers 20-34

## Evaluation Checklist:
- **GT1**: Generalization to a New Model
- **GT2**: Generalization to New Data  
- **GT3**: Method / Specificity Generalizability

In [2]:
# Setup environment
import os
os.chdir('/home/smallyan/eval_agent')

# Load environment variables
import subprocess
result = subprocess.run(['bash', '-c', 'source /home/smallyan/.bashrc && env'], capture_output=True, text=True)
for line in result.stdout.split('\n'):
    if '=' in line:
        key, _, value = line.partition('=')
        if key in ['HF_HOME', 'HF_TOKEN', 'OPENAI_API_KEY', 'NDIF_API_KEY', 'HUGGINGFACE_HUB_CACHE']:
            os.environ[key] = value

import sys
sys.path.insert(0, '/net/scratch2/smallyan/belief_tracking_eval')
sys.path.insert(0, '/net/scratch2/smallyan/belief_tracking_eval/src')

import json
import random
import torch
import numpy as np
from src.dataset import Dataset, Sample

print(f"Working directory: {os.getcwd()}")
print(f"HF_HOME: {os.environ.get('HF_HOME', 'NOT SET')}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Working directory: /home/smallyan/eval_agent
HF_HOME: /net/projects2/chai-lab/shared_models
CUDA available: True
GPU: NVIDIA A40


In [3]:
# Load entity data
data_dir = '/net/scratch2/smallyan/belief_tracking_eval/data'
with open(f'{data_dir}/synthetic_entities/characters.json', 'r') as f:
    all_characters = list(json.load(f))
with open(f'{data_dir}/synthetic_entities/bottles.json', 'r') as f:
    all_objects = list(json.load(f))
with open(f'{data_dir}/synthetic_entities/drinks.json', 'r') as f:
    all_states = list(json.load(f))

print(f"Loaded {len(all_characters)} characters, {len(all_objects)} objects, {len(all_states)} states")

# Create a test sample
random.seed(42)
sample = Sample(
    template_idx=0,
    characters=random.sample(all_characters, 2),
    objects=random.sample(all_objects, 2),
    states=random.sample(all_states, 2),
)
dataset = Dataset(samples=[sample])
item = dataset.__getitem__(0, set_character=0, set_container=0)
print(f"\nSample prompt:\n{item['prompt'][:500]}...")
print(f"\nExpected answer: {item['target']}")

Loaded 103 characters, 21 objects, 23 states

Sample prompt:
Instruction: 1. Track the belief of each character as described in the story. 2. A character's belief is formed only when they perform an action themselves or can observe the action taking place. 3. A character does not have any beliefs about the container and its contents which they cannot observe. 4. To answer the question, predict only what is inside the queried container, strictly based on the belief of the character, mentioned in the question. 5. If the queried character has no belief about...

Expected answer: wine


---
## GT1: Generalization to a New Model

**Objective**: Test if the belief tracking mechanism findings generalize to a model NOT used in the original work.

**New Model**: Mistral-7B-Instruct-v0.3
- Different model family (Mistral vs Llama)
- Not used in original research
- 32 layers (vs 80 layers in Llama-70B)

**Test Approach**:
1. Run causal mediation analysis with interchange interventions
2. Check if information localization patterns scale proportionally to model depth
3. Verify lookback mechanism exists

Expected layer ranges (scaled from 80 to 32 layers):
- Answer payload: layers ~22+ (from 56/80 = 70%)
- Answer pointer: layers ~14-21 (from 34-52/80 = 42-65%)
- Binding: layers ~13-15 (from 33-38/80 = 41-47%)

In [4]:
# Load Mistral model
from nnsight import LanguageModel

hf_cache = '/net/projects2/chai-lab/shared_models/hub'

print("Loading Mistral-7B-Instruct-v0.3...")
mistral_model = LanguageModel(
    'mistralai/Mistral-7B-Instruct-v0.3',
    device_map='cuda',
    dtype=torch.float16,
    dispatch=True,
    cache_dir=hf_cache,
)
print(f"Model loaded! Layers: {len(mistral_model.model.layers)}")

Loading Mistral-7B-Instruct-v0.3...


tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

In [5]:
# Check model loaded
print(f"Model type: {type(mistral_model)}")
num_layers = len(mistral_model.model.layers)
print(f"Number of layers: {num_layers}")

In [6]:
# Test basic print
print("Testing output again...")